In [ ]:
import funciones as f
from PIL import Image
import numpy as np
import pandas as pd

In [ ]:
img_raw = f.cargar_imagen('img/pensamientos.jpg')
h, w, c = img_raw.shape
# img_raw = funciones.cargar_imagen(r"img/flores de lupino.png")
img_cuantizada, paleta_img, labels, colores_paleta = f.cuantizar_imagen(img_raw, n_colores=16)

Image.fromarray(img_cuantizada).save('output/paso2/filtrada_0.png')

img_cuantizada_filtrada, labels_filtrados = f.eliminar_1px(img_cuantizada, labels, colores_paleta)

In [ ]:
def encontrar_vecino_mas_cercano(mapa_regiones, region_id, mascara_region, df_regiones):
    """
    Encuentra la región vecina más grande (preferiblemente del mismo color)
    
    Args:
        mapa_regiones: mapa de regiones actual
        region_id: ID de la región a fusionar
        mascara_region: máscara binaria de la región
        df_regiones: DataFrame con info de regiones
    
    Returns:
        ID de la región vecina más apropiada
    """
    # Dilatar la región para encontrar vecinos
    from scipy.ndimage import binary_dilation
    
    estructura = np.ones((3, 3))  # 8-connectivity
    mascara_dilatada = binary_dilation(mascara_region, structure=estructura)
    
    # Encontrar regiones vecinas (en el borde)
    borde = mascara_dilatada & ~mascara_region
    regiones_vecinas = np.unique(mapa_regiones[borde])
    regiones_vecinas = regiones_vecinas[regiones_vecinas != 0]  # Excluir background
    regiones_vecinas = regiones_vecinas[regiones_vecinas != region_id]  # Excluir a sí misma
    
    if len(regiones_vecinas) == 0:
        return None
    
    # Obtener info de la región actual
    color_actual = df_regiones[df_regiones['region_id'] == region_id]['color_id'].values[0]
    
    # Priorizar vecinos del mismo color, luego por tamaño
    vecinos_info = df_regiones[df_regiones['region_id'].isin(regiones_vecinas)].copy()
    
    # Score: mismo color = +1000000, luego por área
    vecinos_info['score'] = vecinos_info['area_pixels'].copy()
    vecinos_info.loc[vecinos_info['color_id'] == color_actual, 'score'] += 1000000
    
    mejor_vecino = vecinos_info.loc[vecinos_info['score'].idxmax(), 'region_id']
    
    return mejor_vecino

def filtrar_regiones_pequenas(mapa_regiones, df_regiones, area_minima):
    """
    Elimina regiones pequeñas fusionándolas con sus vecinos
    
    Args:
        mapa_regiones: mapa de regiones actual
        df_regiones: DataFrame con info de regiones
        area_minima: umbral de área mínima en píxeles
    
    Returns:
        mapa_regiones_limpio: mapa actualizado
        df_regiones_limpio: DataFrame actualizado
        estadisticas: dict con stats del proceso
    """
    print(f"\n{'='*70}")
    print(f"FILTRANDO REGIONES MENORES A {area_minima} PÍXELES")
    print(f"{'='*70}\n")
    
    # Identificar regiones pequeñas
    regiones_pequenas = df_regiones[df_regiones['area_pixels'] < area_minima].copy()
    regiones_grandes = df_regiones[df_regiones['area_pixels'] >= area_minima].copy()
    
    print(f"Regiones pequeñas a eliminar: {len(regiones_pequenas)}")
    print(f"Regiones que se mantienen: {len(regiones_grandes)}")
    
    if len(regiones_pequenas) == 0:
        print("\n¡No hay regiones pequeñas que filtrar!")
        return mapa_regiones, df_regiones, {'eliminadas': 0, 'fusionadas': 0}
    
    # Ordenar por tamaño (procesar las más pequeñas primero)
    regiones_pequenas = regiones_pequenas.sort_values('area_pixels')
    
    # Crear copia del mapa para modificar
    mapa_limpio = mapa_regiones.copy()
    
    # Tracking de fusiones
    fusiones = {}  # region_pequena -> region_destino
    regiones_eliminadas = 0
    
    print("\nProcesando fusiones...")
    for idx, row in regiones_pequenas.iterrows():
        region_id = row['region_id']
        
        # Crear máscara de esta región
        mascara_region = (mapa_regiones == region_id)
        
        if not mascara_region.any():
            continue
        
        # Encontrar vecino más apropiado
        vecino_id = encontrar_vecino_mas_cercano(mapa_limpio, region_id, mascara_region, df_regiones)
        
        if vecino_id is None:
            print(f"  ⚠ Región {region_id} sin vecinos, se mantiene")
            continue
        
        # Fusionar: reasignar píxeles de region_id a vecino_id
        mapa_limpio[mapa_limpio == region_id] = vecino_id
        fusiones[region_id] = vecino_id
        regiones_eliminadas += 1
        
        if regiones_eliminadas % 50 == 0:
            print(f"  Procesadas {regiones_eliminadas} fusiones...")
    
    print(f"\n✓ Fusiones completadas: {regiones_eliminadas}")
    
    # Reconstruir DataFrame de regiones
    print("\nRecalculando áreas de regiones...")
    nuevas_regiones = []
    
    for region_id in np.unique(mapa_limpio):
        if region_id == 0:
            continue
        
        mascara = (mapa_limpio == region_id)
        area = np.sum(mascara)
        
        # Buscar color original de esta región
        info_original = df_regiones[df_regiones['region_id'] == region_id]
        
        if len(info_original) > 0:
            color_id = info_original.iloc[0]['color_id']
            color_rgb = info_original.iloc[0]['color_rgb']
        else:
            # Esta región fue destino de fusión, mantener su info
            continue
        
        nuevas_regiones.append({
            'region_id': region_id,
            'color_id': color_id,
            'color_rgb': color_rgb,
            'area_pixels': area,
            'porcentaje': 100 * area / mapa_limpio.size
        })
    
    df_limpio = pd.DataFrame(nuevas_regiones)
    
    # Estadísticas
    stats = {
        'eliminadas': regiones_eliminadas,
        'fusionadas': len(fusiones),
        'antes': len(df_regiones),
        'despues': len(df_limpio),
        'reduccion_porcentaje': 100 * (len(df_regiones) - len(df_limpio)) / len(df_regiones)
    }
    
    print(f"\n{'='*70}")
    print(f"RESUMEN DE FILTRADO")
    print(f"{'='*70}")
    print(f"Regiones antes: {stats['antes']}")
    print(f"Regiones después: {stats['despues']}")
    print(f"Reducción: {stats['reduccion_porcentaje']:.1f}%")
    print(f"{'='*70}\n")
    
    return mapa_limpio, df_limpio, stats

In [ ]:
# area minima en base al 0.1% del total de pixeles
def calcular_area_minima(h, w, porcentaje=0.1):
    calc = int((h * w) * (porcentaje / 100))
    return min(calc, 100) 

porcentaje = 0.1
area_minima_est = calcular_area_minima(h, w, porcentaje)



# area_minima = 9
# mapa_regiones, df_regiones = f.segmentar_regiones(
#     img_cuantizada_filtrada,
#     labels_filtrados,
#     colores_paleta
# )

# #filtrar regiones pequeñas
# mapa_regiones_limpio, df_regiones_limpio, stats_filtrado = filtrar_regiones_pequenas(
#     mapa_regiones,
#     df_regiones,
#     area_minima
# )



In [ ]:
area_minima = 1

mapa_regiones, df_regiones = f.segmentar_regiones(
    img_cuantizada_filtrada,
    labels_filtrados,
    colores_paleta
)

# el loop usa el mismo mapa de regiones y los resultados ser recalculan en base al area minima de la ejecucion
while area_minima <= area_minima_est:

    #filtrar regiones pequeñas
    mapa_regiones_limpio, df_regiones_limpio, stats_filtrado = filtrar_regiones_pequenas(
        mapa_regiones,
        df_regiones,
        area_minima
    )

    mapa_colores = dict(zip(df_regiones_limpio['region_id'], df_regiones_limpio['color_id']))
    img_regiones_filtrado = np.vectorize(mapa_colores.get)(mapa_regiones_limpio.reshape(-1))
    img_regiones_filtrado = img_regiones_filtrado.reshape(h, w)

    export = colores_paleta[img_regiones_filtrado - 1]
    Image.fromarray(export).save(f'output/paso3/regiones_filtrada_area_minima_{area_minima}.png')
    area_minima += 1

In [ ]:
# mapa_colores = dict(zip(df_regiones_limpio['region_id'], df_regiones_limpio['color_id']))
# # mapa_regiones_limpio

# img_regiones_filtrado = np.vectorize(mapa_colores.get)(mapa_regiones_limpio.reshape(-1))


In [ ]:
# img_regiones_filtrado = img_regiones_filtrado.reshape(h, w)

In [ ]:
# export = colores_paleta[img_regiones_filtrado - 1]
# Image.fromarray(export).save('output/paso3/regiones_filtrada.png')